# Customer Value Segmentation and 90-Day Repeat-Purchase Propensity

## Data cleaning section

This section loads the parquet file created during the data loading phase (notebook 01),performs basic integrity check for the shape, columns, dtypes and date range. Then it continues with missing-value and duplicate analysis Cancellations/returns and zero/negative quantities and prices and missing customer IDs are then searched for. Afterwards explicit cleaning decisions are made based on the later business/ML objectives, the cleaning based on this decisions is performed and the cleaned dataset is validated.

Finally a cleaned Parquet dataset is created for EDA/feature engineering that follows.



In [1]:
from pathlib import Path

import pandas as pd
import requests


REPO_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

In [2]:
LOCAL_DATA_PATH = (
    REPO_ROOT
    / "data"
    / "interim"
    / "online_retail_raw.parquet"
)

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "apostolis-bloutsos-data/"
    "customer-segmentation-purchase-propensity/"
    "main/data/interim/online_retail_raw.parquet"
)

if LOCAL_DATA_PATH.exists():
    df = pd.read_parquet(LOCAL_DATA_PATH)
    print("Loaded local Parquet file.")

else:
    response = requests.get(DATA_URL, timeout=120)
    response.raise_for_status()

    LOCAL_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    LOCAL_DATA_PATH.write_bytes(response.content)

    df = pd.read_parquet(LOCAL_DATA_PATH)
    print("Downloaded Parquet file from GitHub.")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Downloaded Parquet file from GitHub.
Rows: 1,067,371
Columns: 9


## Initial integrity check

In [3]:
pd.set_option(
    "display.max_columns",
    None
)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 1,067,371
Columns: 9


In [4]:
df.head()

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,source_sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   invoice       1067371 non-null  string        
 1   stock_code    1067371 non-null  string        
 2   description   1062989 non-null  string        
 3   quantity      1067371 non-null  int64         
 4   invoice_date  1067371 non-null  datetime64[ns]
 5   price         1067371 non-null  float64       
 6   customer_id   824364 non-null   float64       
 7   country       1067371 non-null  string        
 8   source_sheet  1067371 non-null  string        
dtypes: datetime64[ns](1), float64(2), int64(1), string(5)
memory usage: 73.3 MB


In [6]:
print("Date range:")
print(df["invoice_date"].min())
print(df["invoice_date"].max())

Date range:
2009-12-01 07:45:00
2011-12-09 12:50:00


## Missing-value analysis

Missing values are quantified before any cleaning decisions are made.
Particular attention is given to customer identifiers and product
descriptions because their treatment may affect downstream customer
segmentation and repeat-purchase propensity modelling.

In [7]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100
}).sort_values("missing_count", ascending=False)

missing_summary

,missing_count,missing_pct
customer_id,243007,22.766873
description,4382,0.410541
invoice,0,0.000000
quantity,0,0.000000
stock_code,0,0.000000
invoice_date,0,0.000000
price,0,0.000000
country,0,0.000000
source_sheet,0,0.000000


The results above show that our focus for missing values should be directed to two features. These are the only features in the dataset that contain missing values: 'customer_id' and 'description'

In [8]:
rows_with_missing = df.isna().any(axis=1).sum()

print(f"Rows with at least one missing value: {rows_with_missing:,}")
print(f"Percentage of rows: {rows_with_missing / len(df) * 100:.2f}%")

Rows with at least one missing value: 243,007
Percentage of rows: 22.77%


In [9]:
# inspection of records with missing values

df[df.isna().any(axis=1)].head(10)

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,source_sheet
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom,Year 2009-2010
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom,Year 2009-2010
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom,Year 2009-2010
470,489521,21646,<NA>,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom,Year 2009-2010
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom,Year 2009-2010
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom,Year 2009-2010
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom,Year 2009-2010
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom,Year 2009-2010
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom,Year 2009-2010
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom,Year 2009-2010


In [10]:
print(f"Missing customer IDs: {df['customer_id'].isna().sum():,}")
print(f"Missing descriptions: {df['description'].isna().sum():,}")

Missing customer IDs: 243,007
Missing descriptions: 4,382


In [11]:
df[df["customer_id"].isna()].head(10)

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,source_sheet
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom,Year 2009-2010
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom,Year 2009-2010
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom,Year 2009-2010
470,489521,21646,<NA>,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom,Year 2009-2010
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom,Year 2009-2010
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom,Year 2009-2010
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom,Year 2009-2010
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom,Year 2009-2010
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom,Year 2009-2010
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom,Year 2009-2010


In [12]:
df[df["description"].isna()].head(10)

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,source_sheet
470,489521,21646,<NA>,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,Year 2009-2010
3114,489655,20683,<NA>,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,Year 2009-2010
3161,489659,21350,<NA>,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom,Year 2009-2010
3731,489781,84292,<NA>,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom,Year 2009-2010
4296,489806,18010,<NA>,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom,Year 2009-2010
4566,489821,85049G,<NA>,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom,Year 2009-2010
6378,489882,35751C,<NA>,12,2009-12-02 16:22:00,0.0,NaN,United Kingdom,Year 2009-2010
6555,489898,79323G,<NA>,954,2009-12-03 09:40:00,0.0,NaN,United Kingdom,Year 2009-2010
6576,489901,21098,<NA>,-200,2009-12-03 09:47:00,0.0,NaN,United Kingdom,Year 2009-2010
6581,489903,21166,<NA>,48,2009-12-03 09:57:00,0.0,NaN,United Kingdom,Year 2009-2010


In [13]:
df.loc[
    df["invoice"] == "489548",
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "invoice_date",
        "price",
        "customer_id",
        "country",
        "source_sheet"
    ]
]

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,source_sheet
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom,Year 2009-2010
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom,Year 2009-2010
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom,Year 2009-2010
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom,Year 2009-2010
1059,489548,22131,FOOD CONTAINER SET 3 LOVE HEART,2,2009-12-01 12:32:00,1.95,NaN,United Kingdom,Year 2009-2010
1060,489548,22079,RIBBON REEL HEARTS DESIGN,10,2009-12-01 12:32:00,1.65,NaN,United Kingdom,Year 2009-2010
1061,489548,22138,BAKING SET 9 PIECE RETROSPOT,3,2009-12-01 12:32:00,4.95,NaN,United Kingdom,Year 2009-2010
1062,489548,22147,FELTCRAFT BUTTERFLY HEARTS,2,2009-12-01 12:32:00,1.45,NaN,United Kingdom,Year 2009-2010
1063,489548,22142,CHRISTMAS CRAFT WHITE FAIRY,2,2009-12-01 12:32:00,1.45,NaN,United Kingdom,Year 2009-2010
1064,489548,22150,3 STRIPEY MICE FELTCRAFT,2,2009-12-01 12:32:00,1.95,NaN,United Kingdom,Year 2009-2010


In [14]:
invoice_customer_audit = (
    df.groupby(["source_sheet", "invoice"])["customer_id"]
      .agg(
          rows="size",
          missing_customer_rows=lambda x: x.isna().sum(),
          known_customer_rows=lambda x: x.notna().sum(),
          unique_known_customers=lambda x: x.dropna().nunique()
      )
      .reset_index()
)

invoice_customer_audit.head()

,source_sheet,invoice,rows,missing_customer_rows,known_customer_rows,unique_known_customers
0,Year 2009-2010,489434,8,0,8,1
1,Year 2009-2010,489435,4,0,4,1
2,Year 2009-2010,489436,19,0,19,1
3,Year 2009-2010,489437,23,0,23,1
4,Year 2009-2010,489438,17,0,17,1


In [15]:
partially_missing_invoices = invoice_customer_audit[
    (invoice_customer_audit["missing_customer_rows"] > 0) &
    (invoice_customer_audit["known_customer_rows"] > 0)
]

print(
    f"Invoices containing both missing and known customer IDs: "
    f"{len(partially_missing_invoices):,}"
)

partially_missing_invoices.head(10)

Invoices containing both missing and known customer IDs: 0


,source_sheet,invoice,rows,missing_customer_rows,known_customer_rows,unique_known_customers


In [16]:
partially_missing_invoices[
    partially_missing_invoices["unique_known_customers"] > 1
]

,source_sheet,invoice,rows,missing_customer_rows,known_customer_rows,unique_known_customers


In [17]:
recoverable_customer_rows = partially_missing_invoices.loc[
    partially_missing_invoices["unique_known_customers"] == 1,
    "missing_customer_rows"
].sum()

print(
    f"Potentially recoverable missing customer-ID rows: "
    f"{recoverable_customer_rows:,}"
)

Potentially recoverable missing customer-ID rows: 0


From the results above the hypothesis of invoices sharing known customer_id and NaN customer_ID and recovering customer_id through shared invoice is rejected and is disproved by the date since Invoices containing both missing and known customer IDs: 0
Potentially recoverable customer-ID rows: 0

For the objectives of this project we infer that these anonymous transactions will ultimately be unusable because both segmentation and repeat-purchase modelling require linking transactions to a specific customer. This though leads us to our first defensible cleaning rule:

customer_id missing
→ cannot assign transaction to a customer
→ exclude from customer-level analytical dataset

Now lets turn our attention to missing descriptions. The question is there a stock code with a missing description that has one consistent known description elsewhere.

In [18]:
stock_description_audit = (
    df.groupby("stock_code")["description"]
      .agg(
          rows="size",
          missing_description_rows=lambda x: x.isna().sum(),
          known_description_rows=lambda x: x.notna().sum(),
          unique_known_descriptions=lambda x: x.dropna().nunique()
      )
      .reset_index()
)

stock_description_audit.head()

,stock_code,rows,missing_description_rows,known_description_rows,unique_known_descriptions
0,10002,400,2,398,1
1,10002R,3,0,3,1
2,10080,31,2,29,2
3,10109,2,1,1,1
4,10120,79,0,79,2


In [19]:
description_candidates = stock_description_audit[
    (stock_description_audit["missing_description_rows"] > 0) &
    (stock_description_audit["known_description_rows"] > 0)
]

print(
    "Stock codes with both missing and known descriptions:",
    len(description_candidates)
)

print(
    "Potentially recoverable missing-description rows:",
    description_candidates["missing_description_rows"].sum()
)

Stock codes with both missing and known descriptions: 2096
Potentially recoverable missing-description rows: 4019


In [20]:
description_candidates[
    description_candidates["unique_known_descriptions"] == 1
].head(20)

,stock_code,rows,missing_description_rows,known_description_rows,unique_known_descriptions
0,10002,400,2,398,1
3,10109,2,1,1,1
5,10123C,71,2,69,1
6,10123G,21,3,18,1
7,10124A,20,1,19,1
12,10134,59,2,57,1
14,10138,28,3,25,1
17,15030,28,1,27,1
19,15036,1020,2,1018,1
22,15044B,194,1,193,1


2,096 stock codes that have both missing and known descriptions.
4,019 missing-description rows could potentially be mapped from a known description for the same stock_code. Many stock codes have exactly one known description, which would make recovery deterministic. On the other hand some stock codes have multiple known descriptions meaning that description mapping would sometimes be ambiguous. The question becomes are the missing-description rows mostly part of the same anonymous/problematic transactions that will later disappear when we exclude missing customer_id.

In [21]:
missing_description_rows = df[df["description"].isna()]

print(f"Total missing descriptions: {len(missing_description_rows):,}")

print(
    "Also missing customer_id:",
    missing_description_rows["customer_id"].isna().sum()
)

print(
    "Percentage also missing customer_id:",
    missing_description_rows["customer_id"].isna().mean() * 100
)

Total missing descriptions: 4,382
Also missing customer_id: 4382
Percentage also missing customer_id: 100.0


In [22]:
missing_description_rows[
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "price",
        "customer_id",
        "country"
    ]
].head(20)

,invoice,stock_code,description,quantity,price,customer_id,country
470,489521,21646,<NA>,-50,0.0,NaN,United Kingdom
3114,489655,20683,<NA>,-44,0.0,NaN,United Kingdom
3161,489659,21350,<NA>,230,0.0,NaN,United Kingdom
3731,489781,84292,<NA>,17,0.0,NaN,United Kingdom
4296,489806,18010,<NA>,-770,0.0,NaN,United Kingdom
4566,489821,85049G,<NA>,-240,0.0,NaN,United Kingdom
6378,489882,35751C,<NA>,12,0.0,NaN,United Kingdom
6555,489898,79323G,<NA>,954,0.0,NaN,United Kingdom
6576,489901,21098,<NA>,-200,0.0,NaN,United Kingdom
6581,489903,21166,<NA>,48,0.0,NaN,United Kingdom


In [23]:
print(
    "Missing-description rows with price = 0:",
    (missing_description_rows["price"] == 0).sum()
)

print(
    "Missing-description rows with quantity <= 0:",
    (missing_description_rows["quantity"] <= 0).sum()
)

Missing-description rows with price = 0: 4382
Missing-description rows with quantity <= 0: 2689


Every one of the 4,382 rows with missing description also has a missing customer_id and price = 0.
Also 2,689 of the rows with missing values have non-positive quantity.

In [24]:
unambiguous_codes = description_candidates[
    description_candidates["unique_known_descriptions"] == 1
]

ambiguous_codes = description_candidates[
    description_candidates["unique_known_descriptions"] > 1
]

print(f"Unambiguous stock codes: {len(unambiguous_codes):,}")
print(f"Ambiguous stock codes: {len(ambiguous_codes):,}")

print(
    "Missing rows recoverable unambiguously:",
    unambiguous_codes["missing_description_rows"].sum()
)

Unambiguous stock codes: 1,616
Ambiguous stock codes: 480
Missing rows recoverable unambiguously: 3062


### Missing-value findings

From the above results we see that missing descriptions can often be reconstructed from stock_code, but doing so provides little value for our customer-level ML objectives, especially since most of those rows are anonymous or non-standard transactions.

This means that the simpler and more defensible decision will be not to impute descriptions at all. That's preferable to adding a transformation simply because it's technically possible but adds no value to our objectives.

Missing customer identifiers cannot be recovered from other line items within
the same invoice: invoices with missing customer IDs contain no known customer
ID elsewhere in the invoice.

All 4,382 rows with missing product descriptions also have missing customer IDs,
and all have a recorded price of zero. Although many descriptions could be
reconstructed from stock codes, these rows remain unusable for the project's
customer-level segmentation and repeat-purchase objectives.

Therefore, product descriptions will not be imputed. Transactions with missing
customer IDs will be excluded later when the final cleaning rules are applied.

## Duplicate Analysis

In [25]:
# Counts how many duplicates are there in the dataset

duplicate_count = df.duplicated().sum()

print(f"Exact duplicate rows: {duplicate_count:,}")
print(f"Percentage of dataset: {duplicate_count / len(df) * 100:.2f}%")

Exact duplicate rows: 12,133
Percentage of dataset: 1.14%


In [26]:
duplicate_rows = df[df.duplicated(keep=False)]

print(f"Rows belonging to duplicate groups: {len(duplicate_rows):,}")

Rows belonging to duplicate groups: 23,430


In [27]:
duplicate_rows.sort_values(
    ["invoice", "invoice_date", "stock_code"]
).head(20)

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,source_sheet
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,Year 2009-2010
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,Year 2009-2010
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom,Year 2009-2010


In [28]:
duplicate_groups = (
    df[df.duplicated(keep=False)]
    .groupby(
        [
            "invoice",
            "stock_code",
            "description",
            "quantity",
            "invoice_date",
            "price",
            "customer_id",
            "country",
            "source_sheet"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="occurrences")
    .sort_values("occurrences", ascending=False)
)

duplicate_groups.head(20)

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,source_sheet,occurrences
8008,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,Year 2010-2011,20
8007,555524,22697,GREEN REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,Year 2010-2011,12
9597,572861,22775,PURPLE DRAWERKNOB ACRYLIC EDWARDIAN,12,2011-10-26 12:46:00,1.25,14102.0,United Kingdom,Year 2010-2011,8
6924,541266,21755,LOVE BUILDING BLOCK WORD,1,2011-01-16 16:25:00,5.95,15673.0,United Kingdom,Year 2010-2011,6
9541,572344,M,Manual,48,2011-10-24 10:43:00,1.50,14607.0,United Kingdom,Year 2010-2011,6
780,496431,84826,ASSTD DESIGN 3D PAPER STICKERS,1,2010-02-01 12:30:00,0.85,16415.0,United Kingdom,Year 2009-2010,6
6869,540524,21756,BATH BUILDING BLOCK WORD,1,2011-01-09 12:53:00,5.95,16735.0,United Kingdom,Year 2010-2011,6
6923,541266,21754,HOME BUILDING BLOCK WORD,1,2011-01-16 16:25:00,5.95,15673.0,United Kingdom,Year 2010-2011,6
6742,538514,21756,BATH BUILDING BLOCK WORD,1,2010-12-12 14:27:00,5.95,15044.0,United Kingdom,Year 2010-2011,6
4055,525065,20894,HANGING BAUBLE T-LIGHT HOLDER LARGE,1,2010-10-03 14:28:00,2.95,16799.0,United Kingdom,Year 2009-2010,6


In [29]:
duplicate_groups["occurrences"].value_counts().sort_index()

,count
occurrences,
2,10625
3,568
4,79
5,12
6,10
8,1
12,1
20,1


In [30]:
print(
    "Unique invoices containing exact duplicate rows:",
    duplicate_rows["invoice"].nunique()
)

print(
    "Unique customers affected:",
    duplicate_rows["customer_id"].nunique()
)

Unique invoices containing exact duplicate rows: 4387
Unique customers affected: 1594


### Exact duplicate rows

An exact duplicate is defined as a transaction row whose values are identical
across all variables in the dataset, including invoice number, stock code,
description, quantity, invoice timestamp, price, customer ID, country and
source period.

`DataFrame.duplicated()` counts repeated copies after the first occurrence.
Therefore, if an identical row occurs three times, two rows are counted as
duplicates. In contrast, `duplicated(keep=False)` identifies all three rows as
members of the duplicate group.

The dataset contains 12,133 duplicate copies (1.14% of all rows). In total,
23,430 observations belong to 11,297 duplicate groups. Most groups contain two
identical observations, although a small number contain substantially more;
the largest exact duplicate group occurs 20 times.

Exact duplication does not necessarily prove that observations are erroneous.
Repeated line items may represent either duplicated records or legitimate
multiple entries of the same product within an invoice. Their relationship
with cancellations and negative quantities is therefore examined before a
cleaning decision is made.

In [31]:
is_duplicate_member = df.duplicated(keep=False)

In [32]:
is_cancelled_invoice = df["invoice"].str.startswith("C", na=False)

In [33]:
duplicate_return_summary = pd.DataFrame({
    "all_rows": [
        len(df),
        (df["quantity"] > 0).sum(),
        (df["quantity"] < 0).sum(),
        (df["quantity"] == 0).sum(),
        is_cancelled_invoice.sum()
    ],
    "rows_in_duplicate_groups": [
        is_duplicate_member.sum(),
        (is_duplicate_member & (df["quantity"] > 0)).sum(),
        (is_duplicate_member & (df["quantity"] < 0)).sum(),
        (is_duplicate_member & (df["quantity"] == 0)).sum(),
        (is_duplicate_member & is_cancelled_invoice).sum()
    ]
}, index=[
    "All rows",
    "Positive quantity",
    "Negative quantity",
    "Zero quantity",
    "Cancellation invoice"
])

duplicate_return_summary["pct_in_duplicate_groups"] = (
    duplicate_return_summary["rows_in_duplicate_groups"]
    / duplicate_return_summary["all_rows"]
    * 100
)

duplicate_return_summary

,all_rows,rows_in_duplicate_groups,pct_in_duplicate_groups
All rows,1067371,23430,2.195113
Positive quantity,1044421,23314,2.232242
Negative quantity,22950,116,0.505447
Zero quantity,0,0,NaN
Cancellation invoice,19494,116,0.595055


In [34]:
# Lets investigate the transaction that occurs 20 times in the dataset as we found earlier when we computed the occurencies of duplicates

df[
    (df["invoice"] == "555524") &
    (df["stock_code"] == "22698")
]

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,source_sheet
738637,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,Year 2010-2011
738638,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,Year 2010-2011
738644,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,Year 2010-2011
738652,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,Year 2010-2011
738653,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,Year 2010-2011
738655,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,Year 2010-2011
738656,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,Year 2010-2011
738657,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,Year 2010-2011
738658,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,Year 2010-2011
738659,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,Year 2010-2011


In [35]:
df[
    df["invoice"] == "555524"
].sort_values("stock_code")

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,source_sheet
738683,555524,20717,STRAWBERRY SHOPPER BAG,3,2011-06-05 11:37:00,1.25,16923.0,United Kingdom,Year 2010-2011
738628,555524,21155,RED RETROSPOT PEG BAG,1,2011-06-05 11:37:00,2.55,16923.0,United Kingdom,Year 2010-2011
738700,555524,21155,RED RETROSPOT PEG BAG,1,2011-06-05 11:37:00,2.55,16923.0,United Kingdom,Year 2010-2011
738680,555524,21166,COOK WITH WINE METAL SIGN,3,2011-06-05 11:37:00,2.08,16923.0,United Kingdom,Year 2010-2011
738681,555524,21174,POTTERING IN THE SHED METAL SIGN,5,2011-06-05 11:37:00,2.08,16923.0,United Kingdom,Year 2010-2011
...,...,...,...,...,...,...,...,...,...
738633,555524,23301,GARDENERS KNEELING PAD KEEP CALM,4,2011-06-05 11:37:00,1.65,16923.0,United Kingdom,Year 2010-2011
738697,555524,82482,WOODEN PICTURE FRAME WHITE FINISH,3,2011-06-05 11:37:00,2.55,16923.0,United Kingdom,Year 2010-2011
738698,555524,82494L,WOODEN FRAME ANTIQUE WHITE,2,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,Year 2010-2011
738629,555524,84978,HANGING HEART JAR T-LIGHT HOLDER,6,2011-06-05 11:37:00,1.25,16923.0,United Kingdom,Year 2010-2011


### Duplicate assessment

The dataset contains 12,133 repeated copies of transaction rows (1.14% of
the full dataset), with 23,430 observations belonging to duplicate groups.

An exact duplicate is defined as a row identical across all available fields,
including invoice, stock code, description, quantity, invoice timestamp,
price, customer ID, country and source period.

Duplicate-group membership is substantially more common among positive
purchase rows (2.23%) than among negative-quantity rows (0.51%) or
cancellation invoices (0.60%). This suggests that exact duplicates are not
primarily associated with returns or cancellations.

Inspection of invoice 555524 also shows repeated product lines within the same
invoice. For example, one product appears 20 times with identical quantity,
price and customer information, while other products in the same invoice
appear repeatedly with both identical and different quantities.

Because the source data does not provide a unique transaction-line identifier,
it is not possible to determine reliably whether identical rows represent
erroneous duplication or legitimate repeated line-item entries. Removing them
could therefore understate purchased quantities and customer monetary value.

Exact duplicate rows are consequently retained.

Just so we know the potential of our decision to keep the duplicate values and to find out how much would our monetary totals change if we had assumed all exact duplicates were erroneous, we will create the line revenue and compare it with a hypothetical exact-deduplicated version.

In [36]:
df["line_revenue"] = df["quantity"] * df["price"]

In [37]:
recorded_revenue = df["line_revenue"].sum()

deduplicated_revenue = (
    df.drop_duplicates()["line_revenue"].sum()
)

difference = recorded_revenue - deduplicated_revenue

print(f"Recorded revenue: £{recorded_revenue:,.2f}")
print(f"Revenue after exact deduplication: £{deduplicated_revenue:,.2f}")
print(f"Difference: £{difference:,.2f}")
print(f"Difference as % of recorded revenue: {difference / recorded_revenue * 100:.2f}%")

Recorded revenue: £19,287,250.57
Revenue after exact deduplication: £19,231,800.52
Difference: £55,450.05
Difference as % of recorded revenue: 0.29%


### Sensitivity of revenue to exact deduplication

To quantify the potential impact of retaining exact duplicate rows, recorded
transaction revenue was compared with a hypothetical version of the dataset
in which exact duplicates were removed.

Removing exact duplicates would reduce total recorded revenue from
£19,287,250.57 to £19,231,800.52, a difference of £55,450.05 or approximately
0.29% of total recorded revenue.

Although the aggregate effect is relatively small, this does not provide a
basis for treating the repeated rows as erroneous. Inspection of individual
invoices showed that repeated identical line items may represent legitimate
transaction entries, and the dataset contains no unique line-item identifier
that would allow true duplicates to be distinguished reliably from valid
repeated purchases.

Furthermore, removing identical lines could have a substantially larger effect
on individual customer-level quantities and monetary values even when the
portfolio-wide revenue impact is small.

Exact duplicate rows are therefore retained. This is the more conservative
choice because it avoids deleting recorded transactions without sufficient
evidence that they are erroneous.

## Cancellations, negative quantities and returns

Negative quantities and cancellation invoices are examined separately before
any transactions are removed or transformed.

In the Online Retail II data, cancellation invoices are identified by invoice
numbers beginning with `C`. However, negative quantities may also occur outside
these invoices. Their relationship is therefore quantified and anomalous cases
are inspected before defining the treatment of returns for customer
segmentation and repeat-purchase modelling.

In [38]:
is_cancellation_invoice = df["invoice"].str.startswith("C", na=False)
is_negative_quantity = df["quantity"] < 0
is_positive_quantity = df["quantity"] > 0

In [39]:
cancellation_summary = pd.DataFrame({
    "rows": [
        len(df),
        is_cancellation_invoice.sum(),
        is_negative_quantity.sum(),
        (is_cancellation_invoice & is_negative_quantity).sum(),
        (is_cancellation_invoice & ~is_negative_quantity).sum(),
        (~is_cancellation_invoice & is_negative_quantity).sum()
    ]
}, index=[
    "All rows",
    "Cancellation-invoice rows",
    "Negative-quantity rows",
    "Cancellation + negative quantity",
    "Cancellation + non-negative quantity",
    "Negative quantity without cancellation invoice"
])

cancellation_summary

,rows
All rows,1067371
Cancellation-invoice rows,19494
Negative-quantity rows,22950
Cancellation + negative quantity,19493
Cancellation + non-negative quantity,1
Negative quantity without cancellation invoice,3457


In [40]:
pd.crosstab(
    is_cancellation_invoice,
    is_negative_quantity,
    rownames=["cancellation_invoice"],
    colnames=["negative_quantity"],
    margins=True
)

negative_quantity,False,True,All
cancellation_invoice,,,
False,1044420,3457,1047877
True,1,19493,19494
All,1044421,22950,1067371


In [41]:
df[
    is_cancellation_invoice & ~is_negative_quantity
][
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "invoice_date",
        "price",
        "customer_id",
        "country"
    ]
].head(20)

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,NaN,United Kingdom


In [42]:
df[
    ~is_cancellation_invoice & is_negative_quantity
][
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "invoice_date",
        "price",
        "customer_id",
        "country"
    ]
].head(30)

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,<NA>,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,<NA>,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom
4296,489806,18010,<NA>,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,NaN,United Kingdom
4566,489821,85049G,<NA>,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom


In [43]:
cancellation_rows = df[is_cancellation_invoice]

print(
    f"Unique cancellation invoices: "
    f"{cancellation_rows['invoice'].nunique():,}"
)

print(
    f"Customers with at least one cancellation: "
    f"{cancellation_rows['customer_id'].nunique():,}"
)

print(
    f"Cancellation rows: "
    f"{len(cancellation_rows):,}"
)

print(
    f"Cancellation rows as % of dataset: "
    f"{len(cancellation_rows) / len(df) * 100:.2f}%"
)

Unique cancellation invoices: 8,292
Customers with at least one cancellation: 2,572
Cancellation rows: 19,494
Cancellation rows as % of dataset: 1.83%


In [44]:
print(
    f"Revenue represented by cancellation invoices: "
    f"£{df.loc[is_cancellation_invoice, 'line_revenue'].sum():,.2f}"
)

print(
    f"Revenue represented by all negative-quantity rows: "
    f"£{df.loc[is_negative_quantity, 'line_revenue'].sum():,.2f}"
)

Revenue represented by cancellation invoices: £-1,526,667.86
Revenue represented by all negative-quantity rows: £-1,527,041.43


In [45]:
negative_non_cancellation = df[
    is_negative_quantity & ~is_cancellation_invoice
].copy()

print(f"Rows: {len(negative_non_cancellation):,}")

print(
    "Missing customer_id:",
    negative_non_cancellation["customer_id"].isna().sum()
)

print(
    "Price = 0:",
    (negative_non_cancellation["price"] == 0).sum()
)

print(
    "Missing customer_id AND price = 0:",
    (
        negative_non_cancellation["customer_id"].isna()
        & (negative_non_cancellation["price"] == 0)
    ).sum()
)

Rows: 3,457
Missing customer_id: 3457
Price = 0: 3457
Missing customer_id AND price = 0: 3457


In [46]:
negative_non_cancellation[
    (negative_non_cancellation["customer_id"].notna()) |
    (negative_non_cancellation["price"] != 0)
][
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "invoice_date",
        "price",
        "customer_id",
        "country"
    ]
].head(30)

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country


In [47]:
negative_non_cancellation["description"].value_counts(
    dropna=False
).head(30)

,count
description,
<NA>,2689
check,123
damages,84
?,83
damaged,78
missing,27
sold as set on dotcom,20
Damaged,17
smashed,9


In [48]:
cancellations = df[is_cancellation_invoice].copy()

print(
    "Cancellation rows with missing customer_id:",
    cancellations["customer_id"].isna().sum()
)

print(
    "Cancellation rows with price = 0:",
    (cancellations["price"] == 0).sum()
)

print(
    "Cancellation rows with positive quantity:",
    (cancellations["quantity"] > 0).sum()
)

Cancellation rows with missing customer_id: 750
Cancellation rows with price = 0: 0
Cancellation rows with positive quantity: 1


### Interpretation of cancellations and negative quantities

Cancellation invoices and negative quantities were examined separately rather
than treating every negative transaction as a customer return.

Of 19,494 rows belonging to invoices beginning with `C`, 19,493 have negative
quantities. The single exception is a manual transaction with a positive
quantity, missing customer identifier and an unusual invoice value.

A further 3,457 rows have negative quantities but do not belong to cancellation
invoices. All 3,457 have a zero price and a missing customer identifier.
Their descriptions frequently contain terms such as `damages`, `missing`,
`smashed`, `thrown away`, `wet damaged` and other stock-control annotations.

These observations are therefore interpreted as internal inventory or
accounting adjustments rather than customer purchase returns.

For downstream customer-level analysis:

- negative non-cancellation adjustments will not be treated as customer
  transactions;
- cancellation invoices with an identifiable customer will be retained as
  return/cancellation behaviour;
- transactions without a customer identifier cannot contribute to customer
  segmentation or repeat-purchase modelling;
- only genuine positive purchase transactions will be eligible to define a
  future repeat purchase for the propensity target.

## Price and transaction-validity analysis

Before constructing the customer-level analytical dataset, transaction prices
and non-standard transaction lines are examined.

The objective is to distinguish genuine merchandise purchases and identifiable
customer cancellations from zero-value records, internal adjustments, postage,
manual entries and other non-product charges. These distinctions are important
because customer value, purchase frequency and the future repeat-purchase target
should reflect genuine purchasing behaviour rather than administrative entries.

In [49]:
price_summary = pd.DataFrame({
    "rows": [
        (df["price"] > 0).sum(),
        (df["price"] == 0).sum(),
        (df["price"] < 0).sum()
    ]
}, index=[
    "Positive price",
    "Zero price",
    "Negative price"
])

price_summary["pct"] = price_summary["rows"] / len(df) * 100

price_summary

,rows,pct
Positive price,1061164,99.418478
Zero price,6202,0.581054
Negative price,5,0.000468


In [50]:
zero_price = df[df["price"] == 0]

print(f"Zero-price rows: {len(zero_price):,}")
print(f"Missing customer_id: {zero_price['customer_id'].isna().sum():,}")
print(f"Known customer_id: {zero_price['customer_id'].notna().sum():,}")
print(f"Positive quantity: {(zero_price['quantity'] > 0).sum():,}")
print(f"Negative quantity: {(zero_price['quantity'] < 0).sum():,}")

Zero-price rows: 6,202
Missing customer_id: 6,131
Known customer_id: 71
Positive quantity: 2,745
Negative quantity: 3,457


In [51]:
df[
    (df["price"] == 0) &
    (df["customer_id"].notna())
][
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "invoice_date",
        "price",
        "customer_id",
        "country"
    ]
].head(30)

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658.0,United Kingdom
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.0,17231.0,United Kingdom
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.0,14108.0,United Kingdom
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.0,14108.0,United Kingdom
32916,492079,85042,ANTIQUE LILY FAIRY LIGHTS,8,2009-12-15 13:49:00,0.0,15070.0,United Kingdom
40101,492760,21143,ANTIQUE GLASS HEART DECORATION,12,2009-12-18 14:22:00,0.0,18071.0,United Kingdom
47126,493761,79320,FLAMINGO LIGHTS,24,2010-01-06 14:54:00,0.0,14258.0,United Kingdom
48342,493899,22355,"CHARLOTTE BAG , SUKI DESIGN",10,2010-01-08 10:43:00,0.0,12417.0,Belgium
57619,494607,21533,RETRO SPOT LARGE MILK JUG,12,2010-01-15 12:43:00,0.0,16858.0,United Kingdom


In [52]:
df[df["price"] < 0][
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "invoice_date",
        "price",
        "customer_id",
        "country"
    ]
]

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [53]:
operational_pattern = (
    r"manual|postage|carriage|bank charge|discount|"
    r"commission|amazon|dotcom|adjust"
)

operational_rows = df[
    df["description"].str.contains(
        operational_pattern,
        case=False,
        na=False,
        regex=True
    )
]

operational_summary = (
    operational_rows
    .groupby(["stock_code", "description"], dropna=False)
    .agg(
        rows=("invoice", "size"),
        customers=("customer_id", "nunique"),
        quantity=("quantity", "sum"),
        revenue=("line_revenue", "sum")
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

operational_summary.head(30)

,stock_code,description,rows,customers,quantity,revenue
129,POST,POSTAGE,2115,496,5158,112341.000
127,DOT,DOTCOM POSTAGE,1444,1,1438,322647.470
128,M,Manual,1421,582,4607,-82796.320
124,C2,CARRIAGE,279,45,266,13386.000
126,D,Discount,177,55,-2872,-13484.540
123,BANK CHARGES,Bank Charges,96,20,-40,-33493.669
65,23444,Next Day Carriage,80,63,78,1185.000
62,23099,FRENCH CARRIAGE LANTERN,63,46,129,872.150
108,85168B,BLACK BAROQUE CARRIAGE CLOCK,53,17,73,662.120
120,AMAZONFEE,AMAZON FEE,43,0,-35,-260763.580


In [54]:
nonstandard_codes = df[
    df["stock_code"].str.fullmatch(r"[A-Za-z ]+", na=False)
]

(
    nonstandard_codes
    .groupby(["stock_code", "description"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
    .head(30)
)

,stock_code,description,rows
22,POST,POSTAGE,2115
17,DOT,DOTCOM POSTAGE,1444
20,M,Manual,1421
8,D,Discount,177
24,S,SAMPLES,104
6,BANK CHARGES,Bank Charges,96
3,AMAZONFEE,AMAZON FEE,43
1,ADJUST,Adjustment by john on 26/01/2010 16,38
2,ADJUST,Adjustment by john on 26/01/2010 17,26
14,DCGSSGIRL,GIRLS PARTY BAG,23


### Price and transaction-validity findings

Almost all transaction rows (99.42%) have a positive unit price. Of the
6,202 zero-price rows, 6,131 have no customer identifier. The remaining
71 identifiable-customer rows do not contribute monetary value and are not
treated as valid purchases for the purposes of customer segmentation or
repeat-purchase modelling.

Only five rows have negative prices. All correspond to bad-debt adjustments,
have no customer identifier and are accounting entries rather than customer
purchases.

Inspection of non-standard stock codes also identified several administrative
or non-merchandise transaction types, including postage and carriage charges,
manual adjustments, discounts, bank charges, commissions, Amazon fees,
samples and test records.

These entries may be relevant for financial reconciliation, but they do not
represent merchandise purchasing behaviour. They are therefore excluded from
the analytical purchase definition used in this project.

A valid purchase line is defined as an identifiable-customer transaction with
a positive quantity and positive price, belonging to a non-cancellation
invoice and representing merchandise rather than an administrative charge.

Identifiable cancellation rows are retained separately because return
behaviour may provide useful customer-level information. Anonymous inventory
adjustments, anonymous cancellations and other records that cannot be linked
to a customer are excluded from the customer-level analytical dataset.

In [55]:
NON_MERCHANDISE_CODES = {
    "M",
    "m",
    "POST",
    "DOT",
    "C2",
    "D",
    "BANK CHARGES",
    "AMAZONFEE",
    "ADJUST",
    "ADJUST2",
    "CRUK",
    "B",
    "S",
    "TEST001",
    "23444"
}

In [56]:
# Defines the business flags
# what is flagged as non-merchandise, purchase and cancellation

df["line_revenue"] = df["quantity"] * df["price"]

df["is_non_merchandise"] = df["stock_code"].isin(
    NON_MERCHANDISE_CODES
)

df["is_cancellation"] = (
    df["customer_id"].notna()
    & df["invoice"].str.startswith("C", na=False)
    & (df["quantity"] < 0)
    & (df["price"] > 0)
    & ~df["is_non_merchandise"]
)

df["is_purchase"] = (
    df["customer_id"].notna()
    & ~df["invoice"].str.startswith("C", na=False)
    & (df["quantity"] > 0)
    & (df["price"] > 0)
    & ~df["is_non_merchandise"]
)

In [57]:
print(f"Raw rows: {len(df):,}")
print(f"Valid purchase rows: {df['is_purchase'].sum():,}")
print(f"Valid cancellation rows: {df['is_cancellation'].sum():,}")

print(
    f"Rows eligible for customer-level analysis: "
    f"{(df['is_purchase'] | df['is_cancellation']).sum():,}"
)

Raw rows: 1,067,371
Valid purchase rows: 802,573
Valid cancellation rows: 17,933
Rows eligible for customer-level analysis: 820,506


In [58]:
print(
    "Purchase revenue:",
    f"£{df.loc[df['is_purchase'], 'line_revenue'].sum():,.2f}"
)

print(
    "Cancellation value:",
    f"£{df.loc[df['is_cancellation'], 'line_revenue'].sum():,.2f}"
)

Purchase revenue: £17,433,265.75
Cancellation value: £-719,674.58


In [59]:
clean_df = df[
    df["is_purchase"] | df["is_cancellation"]
].copy()

In [60]:
clean_df["transaction_type"] = "purchase"

clean_df.loc[
    clean_df["is_cancellation"],
    "transaction_type"
] = "cancellation"

In [61]:
# since we are not going to make calculations with customer_id we store it as string

clean_df["customer_id"] = (
    clean_df["customer_id"]
    .astype("Int64")
    .astype("string")
)

In [62]:
assert clean_df["customer_id"].notna().all()
assert (clean_df["price"] > 0).all()
assert (~clean_df["is_non_merchandise"]).all()

assert (
    clean_df.loc[
        clean_df["transaction_type"] == "purchase",
        "quantity"
    ] > 0
).all()

assert (
    clean_df.loc[
        clean_df["transaction_type"] == "cancellation",
        "quantity"
    ] < 0
).all()

print("Cleaning validation passed.")

Cleaning validation passed.


In [63]:
print(f"Raw rows: {len(df):,}")
print(f"Clean rows: {len(clean_df):,}")
print(f"Rows excluded: {len(df) - len(clean_df):,}")

print()
print(clean_df["transaction_type"].value_counts())

print()
print(f"Customers: {clean_df['customer_id'].nunique():,}")
print(f"Invoices: {clean_df['invoice'].nunique():,}")

Raw rows: 1,067,371
Clean rows: 820,506
Rows excluded: 246,865

transaction_type
purchase        802573
cancellation     17933
Name: count, dtype: int64

Customers: 5,875
Invoices: 43,877


In [64]:
clean_df.isna().sum()

,0
invoice,0
stock_code,0
description,0
quantity,0
invoice_date,0
price,0
customer_id,0
country,0
source_sheet,0
line_revenue,0


In [65]:
CLEAN_DATA_PATH = (
    REPO_ROOT
    / "data"
    / "processed"
    / "online_retail_clean.parquet"
)

CLEAN_DATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

clean_df.to_parquet(
    CLEAN_DATA_PATH,
    index=False
)

print(f"Saved to: {CLEAN_DATA_PATH}")

Saved to: /content/data/processed/online_retail_clean.parquet


In [66]:
test_clean = pd.read_parquet(CLEAN_DATA_PATH)

assert len(test_clean) == len(clean_df)
assert list(test_clean.columns) == list(clean_df.columns)

print("Clean Parquet round-trip validation passed.")

Clean Parquet round-trip validation passed.


## Final cleaning pipeline and analytical dataset

The raw Online Retail II dataset contained 1,067,371 transaction rows. Rather
than applying generic cleaning rules mechanically, each major data-quality issue
was investigated separately so that exclusions could be tied to the project's
customer-segmentation and repeat-purchase objectives.

### Cleaning decisions

The following rules were applied to construct the customer-level analytical
transaction dataset:

- **Transactions without a customer identifier were excluded.**  
  Missing customer IDs could not be recovered reliably from other rows within
  the same invoice. Because both segmentation and repeat-purchase modelling
  require transactions to be linked to a specific customer, anonymous
  transactions cannot contribute to the downstream customer-level analysis.

- **Missing product descriptions were not imputed.**  
  All rows with missing descriptions also had missing customer IDs and were
  therefore already unusable for the customer-level modelling objectives.
  Although many descriptions could theoretically be reconstructed from stock
  codes, doing so would not recover analytically usable customer observations.

- **Exact duplicate rows were retained.**  
  The dataset contains repeated rows that are identical across all available
  transaction fields. However, inspection of individual invoices showed that
  repeated identical line items may represent legitimate repeated entries
  within an order. Because no unique transaction-line identifier exists, true
  erroneous duplicates cannot be distinguished reliably from valid repeated
  purchases.

  Removing exact duplicates would reduce total recorded revenue by only about
  0.29%, but could materially distort quantities and monetary values for
  individual customers. The more conservative decision was therefore to retain
  the recorded transaction lines.

- **Cancellation invoices were distinguished from other negative-quantity
  records.**  
  Almost all invoices beginning with `C` contain negative quantities and are
  interpreted as customer cancellations or returns. Identifiable customer
  cancellations are retained because return behaviour may provide useful
  customer-level information.

- **Negative non-cancellation rows were excluded.**  
  All 3,457 negative-quantity rows without a cancellation invoice had both a
  zero price and a missing customer ID. Their descriptions frequently referred
  to damages, missing stock, destroyed goods, stock checks and similar
  operational adjustments. These records were therefore interpreted as
  inventory or accounting adjustments rather than customer returns.

- **Zero-price transactions were excluded from the valid purchase definition.**  
  A genuine analytical purchase must contribute a positive merchandise value.
  Most zero-price rows were already anonymous, while the small number associated
  with known customers were not treated as valid merchandise purchases.

- **Negative-price accounting entries were excluded.**  
  The five negative-price rows represented bad-debt adjustments rather than
  customer purchases.

- **Administrative and non-merchandise transaction lines were excluded from the
  purchase definition.**  
  Explicit non-merchandise stock codes were identified for items such as
  postage, carriage, manual adjustments, discounts, bank charges, commissions,
  Amazon fees, samples, test records and other accounting entries.

  These entries may be relevant for financial reconciliation, but they do not
  represent merchandise purchasing behaviour and should not determine customer
  purchase frequency, product behaviour or the future repeat-purchase target.

### Analytical transaction definitions

A **valid purchase** is defined as a transaction line that:

1. belongs to an identifiable customer;
2. belongs to a non-cancellation invoice;
3. has a positive quantity;
4. has a positive price; and
5. represents merchandise rather than an administrative or accounting charge.

An **identifiable cancellation** is retained separately when it:

1. belongs to an identifiable customer;
2. belongs to an invoice beginning with `C`;
3. has a negative quantity;
4. has a positive recorded price; and
5. is not classified as a non-merchandise administrative line.

Only genuine positive purchases will later be eligible to define whether a
customer makes a repeat purchase within the future 90-day target window.
Cancellation behaviour may instead be used as an explanatory customer feature.

### Final analytical dataset

After applying these rules:

- Raw transaction rows: **1,067,371**
- Valid purchase rows: **802,573**
- Valid cancellation rows: **17,933**
- Final customer-level transaction rows: **820,506**
- Rows excluded: **246,865**
- Unique customers retained: **5,875**
- Unique invoices retained: **43,877**

The retained purchase transactions represent approximately
**£17.43 million in merchandise purchase value**, while identifiable
cancellation transactions represent approximately **£719.67 thousand in
negative transaction value**.

The resulting dataset therefore preserves the two customer behaviours relevant
to the downstream project:

**merchandise purchasing behaviour** and **identifiable cancellation/return
behaviour**, while removing anonymous, inventory, accounting and
non-merchandise records that cannot meaningfully support customer segmentation
or repeat-purchase modelling.

The cleaned dataset was saved as
`data/processed/online_retail_clean.parquet` and successfully reloaded to verify
that the Parquet round trip preserved the expected rows and schema.